In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('')))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.graph_objects as go
import plotly.subplots as sp
from datetime import datetime

# LUCiD imports
from tools.geometry import generate_detector, load_detector_geom
from tools.simulation import setup_event_simulator
from tools.generate import read_photon_data_from_photonsim
from tools.utils import spherical_to_cartesian, base_dir_path
from tools.optimization.event_cache import get_detector_bounds, generate_random_event_params

## Configuration Parameters

Adjust these parameters to customize the visualization:

In [ ]:
# Configuration parameters
CONFIG = {
    'detector_config': base_dir_path() + 'config/MidBox_geom_config.json',  # Detector configuration file
    'data_file': base_dir_path() + 'data/water/muon/50_data_like_events.root',  # ROOT file with reference photons
    'entry_idx': 2,  # Which entry to use from ROOT file
    'n_photons': 1_000_000,  # Number of photons to simulate
    'K': 6,  # Number of scattering iterations
    'seed': 71900,  # Random seed
    'min_charge': 1.0,  # Minimum charge threshold for display
    'color_by': 'charge',  # Color sensor hits by 'charge' or 'time'
    'dark_theme': False,  # Use dark theme for disc visualizations
    'log_scale': False,  # Use log scale for disc visualizations
    'save_figures': True,  # Save figures to files
    'plot_time': False   # Use time in the 3D plot instead of charge
}

geom_data = load_detector_geom(CONFIG['detector_config'])
CONFIG['detector_type'] = geom_data[0].capitalize()

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Setup Detector and Simulators

In [ ]:
# Setup detector
print("Setting up detector...")
detector = generate_detector(CONFIG['detector_config'])
sensor_positions = jnp.array(detector.all_points)
detector_bounds = get_detector_bounds(detector)
n_sensors = len(sensor_positions)

print(f"  Type: {CONFIG['detector_type']}")
print(f"  Sensors: {n_sensors:,}")
print(f"  Bounds: {detector_bounds}")

# Sensor parameters
sensor_params = (
    jnp.array(50),           # scattering_length
    jnp.array(0.0),         # reflection_rate
    jnp.array(1000),         # absorption_length
    jnp.array(0.001)        # gumbel_softmax_temp
)

In [ ]:
# Setup simulators
print("Setting up simulators...")

# Prediction simulator (regular physics simulation)
prediction_simulator = setup_event_simulator(
    json_filename=CONFIG['detector_config'],
    max_sensors_per_cell=8,
    n_photons=CONFIG['n_photons'],
    temperature=0.05,
    K=CONFIG['K'],
    detector_type=CONFIG['detector_type'],
    is_data=False
)

# Data simulator (transforms reference photons)
data_simulator = setup_event_simulator(
    json_filename=CONFIG['detector_config'],
    max_sensors_per_cell=8,
    n_photons=CONFIG['n_photons'],
    temperature=0.0,  # Zero temperature for data mode
    K=CONFIG['K'],
    detector_type=CONFIG['detector_type'],
    is_data=True
)

print("  Simulators ready")

## Load Reference Photons and Generate Track Parameters

In [ ]:
# Load photon data from ROOT file
print(f"Loading reference photons from ROOT file...")
photon_data = read_photon_data_from_photonsim(CONFIG['data_file'], CONFIG['entry_idx'])
photon_data['N'] = len(photon_data['photon_origins'])

print(f"  Number of photons: {photon_data['N']:,}")
print(f"  Primary energy: {photon_data['energy']:.1f} MeV")

# Generate track parameters
print("\nGenerating track parameters...")

# if you want some random track direction and origin:
# key = jax.random.PRNGKey(CONFIG['seed'])
# track_position, track_direction, _ = generate_random_event_params(key, detector_bounds)

track_position, track_direction = (
    jnp.array([0.0, 0.0, 0.0], dtype=jnp.float32),    # position
    jnp.array([1.0, 1.0, np.sqrt(2)/2], dtype=jnp.float32), 
)


track_energy = photon_data['energy']

print(f"  Position: [{track_position[0]:.3f}, {track_position[1]:.3f}, {track_position[2]:.3f}] m")
print(f"  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]")
print(f"  Energy: {track_energy:.1f} MeV")

## Event Analysis and Statistics

In [ ]:
# Simulate events
print("Simulating events...")
event_key = jax.random.PRNGKey(6)#CONFIG['seed'] + 1000)

# Prediction-like event
print("  Generating prediction-like event...")
# Convert direction for prediction simulator
theta = jnp.arccos(jnp.clip(track_direction[2], -1.0, 1.0))
phi = jnp.arctan2(track_direction[1], track_direction[0])
direction_angles = jnp.array([theta, phi])

prediction_params = (track_energy, track_position, direction_angles)
prediction_charges, prediction_times = prediction_simulator(prediction_params, sensor_params, event_key)

# Data-like event
print("  Generating data-like event...")
data_params = (track_energy, track_position, track_direction)
data_charges, data_times = data_simulator(data_params, sensor_params, event_key, photon_data)

print("  Events generated successfully")

In [ ]:
def analyze_events(prediction_charges, prediction_times, data_charges, data_times, 
                  track_position, track_direction, track_energy, min_charge=5.0):
    """
    Perform quantitative analysis of the two event types.
    """
    # Filter active sensors
    pred_active = prediction_charges > min_charge
    data_active = data_charges > min_charge
    
    pred_charges_active = prediction_charges[pred_active]
    pred_times_active = prediction_times[pred_active]
    data_charges_active = data_charges[data_active]
    data_times_active = data_times[data_active]
    
    print("Event Analysis Summary")
    print("=" * 50)
    print(f"Track Parameters:")
    print(f"  Energy: {track_energy:.1f} MeV")
    print(f"  Position: [{track_position[0]:.3f}, {track_position[1]:.3f}, {track_position[2]:.3f}] m")
    print(f"  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]")
    print()
    
    print(f"Prediction-like Event:")
    print(f"  Active sensors: {len(pred_charges_active):,}")
    print(f"  Total charge: {np.sum(pred_charges_active):.1f}")
    print(f"  Mean charge: {np.mean(pred_charges_active):.2f} ± {np.std(pred_charges_active):.2f}")
    print(f"  Charge range: [{np.min(pred_charges_active):.1f}, {np.max(pred_charges_active):.1f}]")
    print(f"  Mean time: {np.mean(pred_times_active):.1f} ± {np.std(pred_times_active):.1f} ns")
    print(f"  Time range: [{np.min(pred_times_active):.1f}, {np.max(pred_times_active):.1f}] ns")
    print()
    
    print(f"Data-like Event:")
    print(f"  Active sensors: {len(data_charges_active):,}")
    print(f"  Total charge: {np.sum(data_charges_active):.1f}")
    print(f"  Mean charge: {np.mean(data_charges_active):.2f} ± {np.std(data_charges_active):.2f}")
    print(f"  Charge range: [{np.min(data_charges_active):.1f}, {np.max(data_charges_active):.1f}]")
    print(f"  Mean time: {np.mean(data_times_active):.1f} ± {np.std(data_times_active):.1f} ns")
    print(f"  Time range: [{np.min(data_times_active):.1f}, {np.max(data_times_active):.1f}] ns")
    print()
    
    # Comparison
    sensor_ratio = len(data_charges_active) / len(pred_charges_active) if len(pred_charges_active) > 0 else 0
    charge_ratio = np.sum(data_charges_active) / np.sum(pred_charges_active) if np.sum(pred_charges_active) > 0 else 0
    
    print(f"Comparison:")
    print(f"  Active sensor ratio (data/prediction): {sensor_ratio:.2f}")
    print(f"  Total charge ratio (data/prediction): {charge_ratio:.2f}")
    
    # Common sensors
    pred_indices = np.where(pred_active)[0]
    data_indices = np.where(data_active)[0]
    common_active = set(pred_indices) & set(data_indices)
    n_common = len(common_active)
    print(f"  Sensors active in both: {n_common:,}")
    print(f"  Overlap fraction: {n_common / max(len(pred_charges_active), len(data_charges_active)):.2f}")
    
    return pred_indices, data_indices

# Perform analysis
prediction_indices, data_indices = analyze_events(
    prediction_charges, prediction_times, data_charges, data_times,
    track_position, track_direction, track_energy, CONFIG['min_charge']
)

In [ ]:
# Create statistical comparison plots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 6))

# Filter active sensors
pred_active = prediction_charges > CONFIG['min_charge']
data_active = data_charges > CONFIG['min_charge']

pred_charges_active = prediction_charges[pred_active]
pred_times_active = prediction_times[pred_active]
data_charges_active = data_charges[data_active]
data_times_active = data_times[data_active]

# Charge distributions
ax1.hist(pred_charges_active, bins=50, alpha=0.7, label='Prediction-like', color='blue', density=True)
ax1.hist(data_charges_active, bins=50, alpha=0.7, label='Data-like', color='red', density=True)
ax1.set_xlabel('Charge')
ax1.set_ylabel('Density')
ax1.set_title('Charge Distribution Comparison')
ax1.set_yscale("log")#, nonpositive='mask')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Time distributions
ax2.hist(pred_times_active[pred_times_active>0], bins=200, alpha=0.7, label='Prediction-like', color='blue', density=True)
ax2.hist(data_times_active, bins=200, alpha=0.7, label='Data-like', color='red', density=True)
ax2.set_xlabel('Time [ns]')
ax2.set_ylabel('Density')
ax2.set_title('Time Distribution Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Charge vs Time scatter
ax3.scatter(pred_charges_active, pred_times_active, alpha=0.6, s=10, label='Prediction-like', color='blue')
ax3.scatter(data_charges_active, data_times_active, alpha=0.6, s=10, label='Data-like', color='red')
ax3.set_xlabel('Charge')
ax3.set_ylabel('Time [ns]')
ax3.set_title('Charge vs Time Correlation')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Summary statistics
stats_text = f"""
Prediction-like Event:
  Active sensors: {len(pred_charges_active):,}
  Mean charge: {np.mean(pred_charges_active):.2f} ± {np.std(pred_charges_active):.2f}
  Mean time: {np.mean(pred_times_active):.1f} ± {np.std(pred_times_active):.1f} ns

Data-like Event:
  Active sensors: {len(data_charges_active):,}
  Mean charge: {np.mean(data_charges_active):.2f} ± {np.std(data_charges_active):.2f}
  Mean time: {np.mean(data_times_active):.1f} ± {np.std(data_times_active):.1f} ns

Track Parameters:
  Energy: {track_energy:.1f} MeV
  Position: [{track_position[0]:.2f}, {track_position[1]:.2f}, {track_position[2]:.2f}] m
  Direction: [{track_direction[0]:.3f}, {track_direction[1]:.3f}, {track_direction[2]:.3f}]
"""

ax4.text(0.05, 0.95, stats_text, transform=ax4.transAxes, fontsize=10, 
         verticalalignment='top', fontfamily='monospace')
ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1)
ax4.axis('off')
ax4.set_title('Event Statistics')

plt.tight_layout()
plt.show()

if CONFIG['save_figures']:
    figures_dir = Path(base_dir_path()) / 'figures'
    detector_name = Path(CONFIG['detector_config']).stem.replace('_geom_config', '')
    filename = figures_dir / f'{detector_name}_statistical_comparison.png'
    plt.savefig(str(filename), dpi=300, bbox_inches='tight')
    print(f"Saved: {filename}")

## 3D  Visualizations

Use the detector's native visualization:

In [ ]:
# Visualization parameters for disc plots
surface_color = 'black' if CONFIG['dark_theme'] else 'gray'
colorscale_charge = 'inferno' if CONFIG['dark_theme'] else 'viridis'
colorscale_time = 'plasma'

# Convert our dense arrays to sparse format (like load_single_event does)
# The working notebook uses sparse format: indices, charges[indices], times[indices]

# Find non-zero indices for prediction event
pred_nonzero_mask = prediction_charges > 0
pred_sparse_indices = np.where(pred_nonzero_mask)[0]
pred_sparse_charges = prediction_charges[pred_sparse_indices]
pred_sparse_times = prediction_times[pred_sparse_indices]

# Find non-zero indices for data event  
data_nonzero_mask = data_charges > 0
data_sparse_indices = np.where(data_nonzero_mask)[0]
data_sparse_charges = data_charges[data_sparse_indices]
data_sparse_times = data_times[data_sparse_indices]

print("Creating disc-based visualizations...")
print(f"Prediction event: {len(pred_sparse_indices)} sensors with non-zero charge")
print(f"Data event: {len(data_sparse_indices)} sensors with non-zero charge")
print(f"Prediction charges range: [{np.min(pred_sparse_charges):.2f}, {np.max(pred_sparse_charges):.2f}]")
print(f"Data charges range: [{np.min(data_sparse_charges):.2f}, {np.max(data_sparse_charges):.2f}]")
print("\n=== Prediction-like Event - Charge ===\n")

In [ ]:
if CONFIG['save_figures']:
    figures_dir = Path(base_dir_path()) / 'figures'
    detector_name = Path(CONFIG['detector_config']).stem.replace('_geom_config', '')
    if CONFIG['plot_time']:
        filename = figures_dir / f'{detector_name}_time_prediction.pdf'
    else:
        filename = figures_dir / f'{detector_name}_charge_prediction.pdf'
# Prediction event - Charge visualization
# Now use the sparse format exactly like the working notebook
detector.visualize_event_data_plotly_discs(
    pred_sparse_indices, 
    pred_sparse_charges, 
    pred_sparse_times,
    show_all_sensors=True,
    log_scale=CONFIG['log_scale'],
    show_colorbar=False,
    dark_theme=CONFIG['dark_theme'],
    plot_time=CONFIG['plot_time'],
    colorscale=colorscale_charge,
    surface_color=surface_color,
    title="Prediction-like Event - Charge",
    figname=filename
)

In [ ]:
print("\n=== Data-like Event - Charge ===\n")

if CONFIG['save_figures']:
    figures_dir = Path(base_dir_path()) / 'figures'
    detector_name = Path(CONFIG['detector_config']).stem.replace('_geom_config', '')
    if CONFIG['plot_time']:
        filename = figures_dir / f'{detector_name}_time_data_like.pdf'
    else:
        filename = figures_dir / f'{detector_name}_charge_data_like.pdf'

# Data event - Charge visualization
detector.visualize_event_data_plotly_discs(
    data_sparse_indices, 
    data_sparse_charges, 
    data_sparse_times,
    show_all_sensors=True,
    inactive_color='black',
    inactive_opacity=1.0,
    log_scale=CONFIG['log_scale'],
    show_colorbar=False,
    dark_theme=CONFIG['dark_theme'],
    plot_time=CONFIG['plot_time'],
    colorscale=colorscale_charge,
    surface_color=surface_color,
    title="Data-like Event - Charge",
    figname=filename
)